# Erlang C Performance Tests

This notebook measures the performance of the real four-dataset forecasting and scheduling workflow. It runs the expensive forecast only once and reuses the result.

Run every cell from top to bottom. The forecast test can take approximately two minutes. Do not interrupt it unless it runs for an unusually long time.

## Performance limits used

These are simple project testing limits, not universal industry requirements. You may adjust them for your computer.

| Test | Limit |
|---|---:|
| Read four file headers | 5 seconds |
| Full 365-day forecast | 180 seconds |
| Retained memory increase | 2,048 MB |
| Dashboard aggregation | 10 seconds |
| Shift requirements | 5 seconds |
| Headcount calculation | 2 seconds |
| January schedule generation | 10 seconds |
| Average health response | 0.20 seconds |
| Average dashboard response | 0.50 seconds |

In [1]:
import sys
import time
import gc
from pathlib import Path
from datetime import datetime
import pandas as pd
import psutil
from IPython.display import Markdown, display

cwd = Path.cwd()
PROJECT_DIR = cwd if (cwd / 'calculator.py').exists() else cwd.parent
if not (PROJECT_DIR / 'calculator.py').exists():
    raise FileNotFoundError('Could not find calculator.py. Place this notebook inside tests.')
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from calculator import (
    build_stl_forecast, build_dashboard_aggregates,
    build_shift_requirements, calculate_schedule_headcount,
    build_monthly_agent_schedule,
)
from fastapi.testclient import TestClient
from api import app

DATA_CANDIDATES = [cwd / 'data', cwd / 'tests' / 'data', PROJECT_DIR / 'tests' / 'data']
DATA_DIR = next((folder for folder in DATA_CANDIDATES if folder.exists()), DATA_CANDIDATES[-1])
CDR_FILES = [DATA_DIR / f'cdr_{year}.csv' for year in range(2021, 2025)]
LIMITS = {
    'read_seconds': 5, 'forecast_seconds': 180, 'memory_mb': 2048,
    'dashboard_seconds': 10, 'requirements_seconds': 5,
    'headcount_seconds': 2, 'schedule_seconds': 10,
    'health_seconds': 0.20, 'page_seconds': 0.50,
}
PERFORMANCE_RESULTS = []
STATE = {}

def record(test_id, description, passed, limit, actual, details=''):
    PERFORMANCE_RESULTS[:] = [row for row in PERFORMANCE_RESULTS if row['test'] != test_id]
    status = 'PASS' if passed else 'FAIL'
    PERFORMANCE_RESULTS.append({'test': test_id, 'description': description, 'limit': str(limit), 'actual': str(actual), 'status': status, 'details': details})
    print(f'{status}: {test_id} — {description}')
    print('Limit: ', limit)
    print('Actual:', actual)
    if details: print('Details:', details)
    return passed

print('Setup complete')
print('Python:', sys.executable)
print('Project directory:', PROJECT_DIR)
print('Dataset directory:', DATA_DIR)

Setup complete
Python: c:\Users\adept\Desktop\ADEPT\ERLANG\Calculator-Erlang-C\.venv\Scripts\python.exe
Project directory: c:\Users\adept\Desktop\ADEPT\ERLANG\Calculator-Erlang-C
Dataset directory: c:\Users\adept\Desktop\ADEPT\ERLANG\Calculator-Erlang-C\tests\data


c:\Users\adept\Desktop\ADEPT\ERLANG\Calculator-Erlang-C\.venv\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


## PERF-01 — Dataset access time
Checks that all four files exist and their headers can be read quickly.

In [2]:
missing = [path.name for path in CDR_FILES if not path.exists()]
started = time.perf_counter()
error = ''
if not missing:
    try:
        for path in CDR_FILES:
            pd.read_csv(path, nrows=5)
    except Exception as exc:
        error = f'{type(exc).__name__}: {exc}'
elapsed = time.perf_counter() - started
record('PERF-01', 'Four dataset headers are read', not missing and not error and elapsed <= LIMITS['read_seconds'], f"<= {LIMITS['read_seconds']} seconds", f'{elapsed:.3f} seconds', f'missing={missing}; error={error}')

PASS: PERF-01 — Four dataset headers are read
Limit:  <= 5 seconds
Actual: 0.050 seconds
Details: missing=[]; error=


True

## PERF-02 and PERF-03 — Full forecast time and memory
Uses all four historical datasets to produce the complete 365-day forecast. Memory is the retained process-memory change, not an exact peak-memory measurement.

In [3]:
if any(not path.exists() for path in CDR_FILES):
    record('PERF-02', 'Full four-year forecast execution time', False, f"<= {LIMITS['forecast_seconds']} seconds", 'Datasets missing')
    record('PERF-03', 'Retained memory increase during forecast', False, f"<= {LIMITS['memory_mb']} MB", 'Datasets missing')
else:
    gc.collect()
    process = psutil.Process()
    memory_before = process.memory_info().rss / (1024 * 1024)
    started = time.perf_counter()
    try:
        forecast, forecast_summary = build_stl_forecast(
            file_paths=CDR_FILES, filenames=[path.name for path in CDR_FILES],
            interval_minutes=30, forecast_days=365, trend_lookback_days=90,
            target_seconds=20, target_service_level=80, shrinkage=30, max_agents=10000,
        )
        elapsed = time.perf_counter() - started
        memory_after = process.memory_info().rss / (1024 * 1024)
        memory_increase = max(memory_after - memory_before, 0)
        STATE['forecast'] = forecast
        STATE['forecast_summary'] = forecast_summary
        STATE['forecast_seconds'] = elapsed
        STATE['memory_increase_mb'] = memory_increase
        record('PERF-02', 'Full four-year forecast execution time', elapsed <= LIMITS['forecast_seconds'], f"<= {LIMITS['forecast_seconds']} seconds", f'{elapsed:.3f} seconds', f'Rows produced: {len(forecast):,}')
        record('PERF-03', 'Retained memory increase during forecast', memory_increase <= LIMITS['memory_mb'], f"<= {LIMITS['memory_mb']} MB", f'{memory_increase:.2f} MB', f'Before={memory_before:.2f} MB; after={memory_after:.2f} MB')
    except Exception as exc:
        elapsed = time.perf_counter() - started
        message = f'{type(exc).__name__}: {exc}'
        record('PERF-02', 'Full four-year forecast execution time', False, f"<= {LIMITS['forecast_seconds']} seconds", message, f'Elapsed before failure: {elapsed:.3f} seconds')
        record('PERF-03', 'Retained memory increase during forecast', False, f"<= {LIMITS['memory_mb']} MB", 'Forecast failed')

PASS: PERF-02 — Full four-year forecast execution time
Limit:  <= 180 seconds
Actual: 111.456 seconds
Details: Rows produced: 17,520
PASS: PERF-03 — Retained memory increase during forecast
Limit:  <= 2048 MB
Actual: 253.05 MB
Details: Before=194.52 MB; after=447.57 MB


## PERF-04 — Forecast output size
Checks that a 365-day forecast with 30-minute intervals contains exactly 17,520 rows.

In [4]:
if 'forecast' not in STATE:
    record('PERF-04', 'Forecast output row count', False, '17,520 rows', 'Forecast unavailable')
else:
    row_count = len(STATE['forecast'])
    record('PERF-04', 'Forecast output row count', row_count == 365 * 48, '17,520 rows', f'{row_count:,} rows')

PASS: PERF-04 — Forecast output row count
Limit:  17,520 rows
Actual: 17,520 rows


## PERF-05 to PERF-08 — Dashboard and scheduling performance
Measures each stage separately while reusing the forecast.

In [5]:
if 'forecast' not in STATE:
    for test_id, description, key in [
        ('PERF-05', 'Dashboard aggregation time', 'dashboard_seconds'),
        ('PERF-06', 'January shift-requirement time', 'requirements_seconds'),
        ('PERF-07', 'Headcount calculation time', 'headcount_seconds'),
        ('PERF-08', 'January schedule generation time', 'schedule_seconds'),
    ]:
        record(test_id, description, False, f"<= {LIMITS[key]} seconds", 'Forecast unavailable')
else:
    started = time.perf_counter(); charts = build_dashboard_aggregates(STATE['forecast']); dashboard_time = time.perf_counter() - started
    record('PERF-05', 'Dashboard aggregation time', dashboard_time <= LIMITS['dashboard_seconds'], f"<= {LIMITS['dashboard_seconds']} seconds", f'{dashboard_time:.3f} seconds')
    started = time.perf_counter(); requirements = build_shift_requirements(STATE['forecast'], year=2025, month=1); requirements_time = time.perf_counter() - started
    record('PERF-06', 'January shift-requirement time', requirements_time <= LIMITS['requirements_seconds'], f"<= {LIMITS['requirements_seconds']} seconds", f'{requirements_time:.3f} seconds', f'Rows: {len(requirements)}')
    started = time.perf_counter(); headcount = calculate_schedule_headcount(requirements); headcount_time = time.perf_counter() - started
    record('PERF-07', 'Headcount calculation time', headcount_time <= LIMITS['headcount_seconds'], f"<= {LIMITS['headcount_seconds']} seconds", f'{headcount_time:.3f} seconds', f'Headcount: {headcount}')
    started = time.perf_counter(); schedule, schedule_summary = build_monthly_agent_schedule(STATE['forecast'], year=2025, month=1, agent_count=None); schedule_time = time.perf_counter() - started
    STATE['schedule'] = schedule
    STATE['schedule_summary'] = schedule_summary
    record('PERF-08', 'January schedule generation time', schedule_time <= LIMITS['schedule_seconds'], f"<= {LIMITS['schedule_seconds']} seconds", f'{schedule_time:.3f} seconds', f'Rows: {len(schedule):,}; agents: {schedule_summary.get("agent_count")}')

PASS: PERF-05 — Dashboard aggregation time
Limit:  <= 10 seconds
Actual: 0.181 seconds
PASS: PERF-06 — January shift-requirement time
Limit:  <= 5 seconds
Actual: 0.037 seconds
Details: Rows: 93
PASS: PERF-07 — Headcount calculation time
Limit:  <= 2 seconds
Actual: 0.003 seconds
Details: Headcount: 48
PASS: PERF-08 — January schedule generation time
Limit:  <= 10 seconds
Actual: 0.063 seconds
Details: Rows: 1,488; agents: 48


## PERF-09 and PERF-10 — Basic API response times
Sends 20 local requests using FastAPI TestClient. No external server is required.

In [6]:
client = TestClient(app)
def average_response_time(path, repeats=20):
    times = []
    statuses = []
    for _ in range(repeats):
        started = time.perf_counter()
        response = client.get(path)
        times.append(time.perf_counter() - started)
        statuses.append(response.status_code)
    return sum(times) / len(times), statuses

health_average, health_statuses = average_response_time('/health')
record('PERF-09', 'Average health API response time', all(code == 200 for code in health_statuses) and health_average <= LIMITS['health_seconds'], f"<= {LIMITS['health_seconds']} seconds", f'{health_average:.4f} seconds', '20 local requests')
page_average, page_statuses = average_response_time('/')
record('PERF-10', 'Average dashboard response time', all(code == 200 for code in page_statuses) and page_average <= LIMITS['page_seconds'], f"<= {LIMITS['page_seconds']} seconds", f'{page_average:.4f} seconds', '20 local requests')

PASS: PERF-09 — Average health API response time
Limit:  <= 0.2 seconds
Actual: 0.0060 seconds
Details: 20 local requests
PASS: PERF-10 — Average dashboard response time
Limit:  <= 0.5 seconds
Actual: 0.0051 seconds
Details: 20 local requests


True

## Final performance report

In [7]:
expected_ids = [f'PERF-{number:02d}' for number in range(1, 11)]
completed = {row['test'] for row in PERFORMANCE_RESULTS}
passed = sum(row['status'] == 'PASS' for row in PERFORMANCE_RESULTS)
failed = sum(row['status'] == 'FAIL' for row in PERFORMANCE_RESULTS)
not_run = [test_id for test_id in expected_ids if test_id not in completed]
table_rows = []
for test_id in expected_ids:
    row = next((item for item in PERFORMANCE_RESULTS if item['test'] == test_id), None)
    if row:
        table_rows.append(f"| {row['test']} | {row['description']} | {row['limit']} | {row['actual']} | {row['status']} |")
    else:
        table_rows.append(f'| {test_id} | Not run | — | — | NOT RUN |')
final_status = 'PASS' if passed == len(expected_ids) else ('FAIL' if failed else 'INCOMPLETE')
report = f'''
# Erlang C Performance Testing Report

## Test information

- Test date: {datetime.now().strftime('%Y-%m-%d %H:%M')}
- Historical datasets: 2021, 2022, 2023 and 2024
- Forecast duration: 365 days
- Forecast interval: 30 minutes
- Computer-specific limits: Yes

## Overall results

| Result | Count |
|---|---:|
| Expected tests | {len(expected_ids)} |
| Passed | {passed} |
| Failed | {failed} |
| Not run | {len(not_run)} |

## Detailed results

| Test | Description | Limit | Actual | Result |
|---|---|---|---|---|
{chr(10).join(table_rows)}

## Final status

{final_status}

{'All performance checks passed.' if final_status == 'PASS' else ('One or more performance checks exceeded the selected limits.' if final_status == 'FAIL' else 'Run all cells before finalizing.')}
'''
display(Markdown(report))
report_path = PROJECT_DIR / 'tests' / 'Erlang_C_Performance_Test_Report.md'
report_path.write_text(report, encoding='utf-8')
print('Report saved to:', report_path)


# Erlang C Performance Testing Report

## Test information

- Test date: 2026-09-05 01:53
- Historical datasets: 2021, 2022, 2023 and 2024
- Forecast duration: 365 days
- Forecast interval: 30 minutes
- Computer-specific limits: Yes

## Overall results

| Result | Count |
|---|---:|
| Expected tests | 10 |
| Passed | 10 |
| Failed | 0 |
| Not run | 0 |

## Detailed results

| Test | Description | Limit | Actual | Result |
|---|---|---|---|---|
| PERF-01 | Four dataset headers are read | <= 5 seconds | 0.050 seconds | PASS |
| PERF-02 | Full four-year forecast execution time | <= 180 seconds | 111.456 seconds | PASS |
| PERF-03 | Retained memory increase during forecast | <= 2048 MB | 253.05 MB | PASS |
| PERF-04 | Forecast output row count | 17,520 rows | 17,520 rows | PASS |
| PERF-05 | Dashboard aggregation time | <= 10 seconds | 0.181 seconds | PASS |
| PERF-06 | January shift-requirement time | <= 5 seconds | 0.037 seconds | PASS |
| PERF-07 | Headcount calculation time | <= 2 seconds | 0.003 seconds | PASS |
| PERF-08 | January schedule generation time | <= 10 seconds | 0.063 seconds | PASS |
| PERF-09 | Average health API response time | <= 0.2 seconds | 0.0060 seconds | PASS |
| PERF-10 | Average dashboard response time | <= 0.5 seconds | 0.0051 seconds | PASS |

## Final status

PASS

All performance checks passed.


Report saved to: c:\Users\adept\Desktop\ADEPT\ERLANG\Calculator-Erlang-C\tests\Erlang_C_Performance_Test_Report.md
